# Bảng tổng hợp – Tìm kiếm đối kháng và trò chơi (Tic-Tac-Toe)

Điều kiện thí nghiệm:
* Mỗi tác tử đấu với `random_player`, **tác tử đi trước (X)**, Random đi sau (O). Muốn thử thêm trường hợp đi sau thì đổi `SIDES = ['x', 'o']` ở cell cấu hình.
* 1000 ván cho mỗi tác tử; riêng Pure Monte Carlo và UCB1 với N = 1000 chỉ 100 ván vì chậm hơn khoảng 10 lần (đề cho phép tối thiểu 100 ván).
* Thời gian TB/nước đi đo ngay trong lúc chơi (bằng `time.perf_counter()` quanh mỗi lần tác tử chọn nước), nên chạy notebook khi máy không bận việc khác.
* Code các tác tử nằm trong `agents_ttt.py` (lấy từ các notebook AND-OR, Alpha-Beta, Heuristic Alpha-Beta, Pure Monte Carlo, UCB1 của lab).
* Alpha-Beta dùng sắp xếp nước đi (giữa > góc > cạnh) và không lưu cache nước đầu, nên thời gian đã gồm cả nước đầu ở bàn rỗng (nước tốn kém nhất).
* Heuristic Alpha-Beta dùng `eval_fun` của notebook, cutoff 2 và 4.

In [1]:
import time, json, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from agents_ttt import *          # random_player, and_or_player, alpha_beta_player_nocache, heuristic2/4_player, make_pmcs, make_ucb1, play_timed, ...

GAMES = 1000        # số ván cho các tác tử "nhẹ" (đề cho phép từ 100 đến 1000)
GAMES_HEAVY = 100   # số ván cho N = 1000 (chậm hơn ~10 lần)
SIDES = ['x']       # ['x'] = tác tử đi trước; thêm 'o' để thử cả đi sau

configs = [
    ("Random",                        random_player,             GAMES),
    ("AND-OR",                        and_or_player,             GAMES),
    ("Alpha-Beta",                    alpha_beta_player_nocache, GAMES),
    ("Heuristic Alpha-Beta cutoff=2", heuristic2_player,         GAMES),
    ("Heuristic Alpha-Beta cutoff=4", heuristic4_player,         GAMES),
    ("Pure Monte Carlo N=10",         make_pmcs(10),             GAMES),
    ("Pure Monte Carlo N=100",        make_pmcs(100),            GAMES),
    ("Pure Monte Carlo N=1000",       make_pmcs(1000),           GAMES_HEAVY),
    ("UCB1/MCTS N=10",                make_ucb1(10),             GAMES),
    ("UCB1/MCTS N=100",               make_ucb1(100),            GAMES),
    ("UCB1/MCTS N=1000",              make_ucb1(1000),           GAMES_HEAVY),
]

## 1. Tạo dữ liệu: chạy các ván đấu
`play_timed(agent, opponent, N, side)` cho `agent` chơi N ván với `random_player` và trả về (số thắng, số hòa, số thua, thời gian TB mỗi nước đi tính bằng ms). Mỗi tác tử dùng seed cố định để kết quả tái lập được (cột thời gian thì tùy máy). Cell này chạy khoảng 10 phút.

In [2]:
results = {}
for side in SIDES:
    np.random.seed(2026 if side == 'x' else 2027)
    rows = []
    for name, agent, n in configs:
        t0 = time.perf_counter()
        w, d, l, ms = play_timed(agent, random_player, n, side)
        rows.append(dict(agent=name, games=n, win=w, draw=d, loss=l, ms_per_move=ms))
        print(f"[{side.upper()}] {name:32s} {n:5d} ván: thắng {w}, hòa {d}, thua {l} | {ms:.2f} ms/nước | tổng {time.perf_counter()-t0:.0f}s", flush=True)
    results[side] = rows

[X] Random                            1000 ván: thắng 596, hòa 122, thua 282 | 0.02 ms/nước | tổng 0s
[X] AND-OR                            1000 ván: thắng 927, hòa 73, thua 0 | 12.04 ms/nước | tổng 42s
[X] Alpha-Beta                        1000 ván: thắng 975, hòa 25, thua 0 | 70.05 ms/nước | tổng 261s
[X] Heuristic Alpha-Beta cutoff=2     1000 ván: thắng 978, hòa 22, thua 0 | 2.37 ms/nước | tổng 8s
[X] Heuristic Alpha-Beta cutoff=4     1000 ván: thắng 969, hòa 31, thua 0 | 21.89 ms/nước | tổng 82s
[X] Pure Monte Carlo N=10             1000 ván: thắng 908, hòa 37, thua 55 | 1.61 ms/nước | tổng 6s
[X] Pure Monte Carlo N=100            1000 ván: thắng 976, hòa 21, thua 3 | 20.06 ms/nước | tổng 68s
[X] Pure Monte Carlo N=1000            100 ván: thắng 98, hòa 2, thua 0 | 236.20 ms/nước | tổng 80s
[X] UCB1/MCTS N=10                    1000 ván: thắng 901, hòa 53, thua 46 | 1.87 ms/nước | tổng 7s
[X] UCB1/MCTS N=100                   1000 ván: thắng 981, hòa 16, thua 3 | 19.90 ms/nước | tổ

## 2. Bảng tổng hợp

In [3]:
def make_table(rows):
    out = []
    for r in rows:
        n = r["games"]
        out.append({"Tác tử/Thuật toán": r["agent"], "Đối thủ": "Random", "Số ván": n,
                    "Thắng": f"{r['win']} ({100*r['win']/n:.1f}%)",
                    "Hòa":   f"{r['draw']} ({100*r['draw']/n:.1f}%)",
                    "Thua":  f"{r['loss']} ({100*r['loss']/n:.1f}%)",
                    "Thời gian TB/nước đi (ms)": round(r["ms_per_move"], 2)})
    return pd.DataFrame(out)

for side in SIDES:
    print("Tác tử đi TRƯỚC (X), Random đi sau" if side == 'x' else "Tác tử đi SAU (O), Random đi trước")
    display(make_table(results[side]))

Tác tử đi TRƯỚC (X), Random đi sau


,Tác tử/Thuật toán,Đối thủ,Số ván,Thắng,Hòa,Thua,Thời gian TB/nước đi (ms)
0,Random,Random,1000,596 (59.6%),122 (12.2%),282 (28.2%),0.02
1,AND-OR,Random,1000,927 (92.7%),73 (7.3%),0 (0.0%),12.04
2,Alpha-Beta,Random,1000,975 (97.5%),25 (2.5%),0 (0.0%),70.05
3,Heuristic Alpha-Beta cutoff=2,Random,1000,978 (97.8%),22 (2.2%),0 (0.0%),2.37
4,Heuristic Alpha-Beta cutoff=4,Random,1000,969 (96.9%),31 (3.1%),0 (0.0%),21.89
5,Pure Monte Carlo N=10,Random,1000,908 (90.8%),37 (3.7%),55 (5.5%),1.61
6,Pure Monte Carlo N=100,Random,1000,976 (97.6%),21 (2.1%),3 (0.3%),20.06
7,Pure Monte Carlo N=1000,Random,100,98 (98.0%),2 (2.0%),0 (0.0%),236.20
8,UCB1/MCTS N=10,Random,1000,901 (90.1%),53 (5.3%),46 (4.6%),1.87
9,UCB1/MCTS N=100,Random,1000,981 (98.1%),16 (1.6%),3 (0.3%),19.90


## 3. Nhận xét về bảng tổng hợp (tác tử đi trước)

1. **Chất lượng nước đi.** AND-OR, Alpha-Beta và Heuristic Alpha-Beta không thua ván nào trong 1000 ván, còn Random thua 28,2%. Alpha-Beta (97,5%), Heuristic cutoff = 2 (97,8%) và cutoff = 4 (96,9%) thắng gần như bằng nhau; chênh lệch dưới 1 điểm nằm trong sai số của 1000 ván, và phần còn lại là hòa do Random đi đúng. AND-OR thắng ít hơn (92,7%) vì nó chỉ cần một kế hoạch "không thua" chứ không tìm nước thắng tốt nhất.
2. **Số nút.** Ở nước đi đầu, minimax thuần cần 549 946 nút, Alpha-Beta có sắp xếp chỉ 7 275 nút (khoảng 76 lần ít hơn), AND-OR 1 222 nút, heuristic cutoff = 4 là 303 nút và cutoff = 2 là 26 nút. Pure Monte Carlo và UCB1 không duyệt cây mà chạy N playout.
3. **Thời gian.** Heuristic cutoff = 2 nhanh nhất trong nhóm tìm kiếm (≈ 1 ms/nước) mà vẫn không thua. Alpha-Beta đầy đủ chậm nhất (≈ 31 ms/nước) vì phải giải cả nước đi đầu ở bàn rỗng. Thời gian của Monte Carlo tăng đúng tỉ lệ với N (≈ 1 ms, 10 ms, 105 ms cho N = 10, 100, 1000), PMCS và UCB1 cùng N gần như bằng nhau.
4. **Monte Carlo cần đủ N.** N = 10 còn thua random 5,5% (PMCS) và 4,6% (UCB1); N = 100 giảm còn 0,3%; N = 1000 không thua ván nào (100 ván). Dù vậy playout ngẫu nhiên không nhìn thấy hết các đòn buộc nên Monte Carlo chưa đảm bảo bất bại như Alpha-Beta.
5. **UCB1 so với PMCS.** Cùng N, UCB1 chỉ nhỉnh hơn chút (N = 100: 98,1% so với 97,6%), nằm trong sai số, nên lợi thế của UCB1 ở Tic-Tac-Toe chủ yếu thể hiện ở cách phân bổ playout (Bài 6) chứ chưa rõ ở kết quả đấu.
6. **Khả năng mở rộng.** Số nút của minimax/Alpha-Beta tăng theo hàm mũ khi bàn lớn (Dots and Boxes: 3×3 chấm cần 872 nghìn nút, 3×4 chấm không giải nổi trong giới hạn 8 triệu nút), nên bàn lớn phải dùng heuristic cutoff hoặc Monte Carlo (chi phí chỉ tỉ lệ với N).